<a href="https://colab.research.google.com/github/mxrch5th/2026-Visionaries-Kiosk/blob/main/recog_faces_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pickle
import numpy as np
import os
import sys
from google.colab import files

# DeepFace와 머신러닝 라이브러리 자동 설치 및 로드
if 'deepface' not in sys.modules:
    print("🔄 DeepFace 라이브러리를 설치하고 있습니다. (약 10~20초 소요)")
    !pip install deepface -q
from deepface import DeepFace
from sklearn.linear_model import LogisticRegression

# 💡 내 드라이브 안의 pkl 파일 경로를 체크하는 로직 보완
# (MyDrive 대소문자 에러 방지용)
possible_paths = [
    '/content/drive/MyDrive/kiosk_ai_model.pkl',
    '/content/drive/mydrive/kiosk_ai_model.pkl',
    '/content/kiosk_ai_model.pkl'  # 혹시 코랩 메인에 그냥 있을 경우
]

model_filename = None
for path in possible_paths:
    if os.path.exists(path):
        model_filename = path
        break

# 만약 위 경로에 모두 없다면 파일 이름을 직접 검색해 봅니다.
if model_filename is None:
    for root, dirs, files_list in os.walk('/content/drive'):
        if 'kiosk_ai_model.pkl' in files_list:
            model_filename = os.path.join(root, 'kiosk_ai_model.pkl')
            break

# 개수 기록용 파일 경로 설정 (모델이 있는 폴더와 같은 곳에 저장)
if model_filename:
    count_filename = os.path.join(os.path.dirname(model_filename), 'model_data_count.txt')
else:
    # 정말 못 찾았을 때를 위한 기본 경로 세팅
    model_filename = '/content/drive/MyDrive/kiosk_ai_model.pkl'
    count_filename = '/content/drive/MyDrive/model_data_count.txt'

# =====================================================================
# [확인 단계] 기존 드라이브에 있던 pkl 파일과 누적 개수 가져오기
# =====================================================================
current_total_count = 0

if os.path.exists(model_filename):
    with open(model_filename, 'rb') as file:
        kiosk_ai_model = pickle.load(file)

    # 예전에 저장해둔 장수 기록 파일이 있으면 읽어오고, 없으면 로그 기준으로 '70장' 세팅
    if os.path.exists(count_filename):
        with open(count_filename, 'r') as f:
            current_total_count = int(f.read().strip())
    else:
        current_total_count = 70

    print("==================================================")
    print("🎯 [기존 모델 불러오기 성공]")
    print(f"📂 파일 위치: {model_filename}")
    print(f"📊 현재 이 AI 모델에 누적 저장된 기존 사진 개수: {current_total_count}장")
    print("==================================================")
else:
    print("❌ 에러: 구글 드라이브에서 'kiosk_ai_model.pkl' 파일을 찾을 수 없습니다.")
    print("💡 코랩 왼쪽 메뉴에서 [📁 폴더 아이콘]을 눌러 드라이브 안에 파일이 진짜 있는지, 이름이 정확한지 확인해 주세요!")
    sys.exit()

# =====================================================================
# [수집 단계] 추가할 71번째 사진부터 등록하기
# =====================================================================
X_new_data = []
y_new_data = []

print("\n==================================================")
print(f"▶ 기존 {current_total_count}장에 이어서 추가할 [새 사진]을 등록합니다.")
print("  (추가할 사진을 모두 올렸다면 '취소'나 '창닫기'를 누르세요)")
print("==================================================")

while True:
    uploaded = files.upload()
    if not uploaded:
        print(f"\n▶ 사진 등록 완료! 이번에 총 {len(X_new_data)}장의 사진이 새로 추가되었습니다.")
        break

    for original_name in uploaded.keys():
        temp_english_name = "temp_kiosk_face_image.png"
        try:
            if os.path.exists(original_name):
                os.rename(original_name, temp_english_name)

            # 새 사진 특징값 추출
            embedding_objs = DeepFace.represent(img_path=temp_english_name, model_name="VGG-Face", enforce_detection=False)
            face_feature = embedding_objs[0]["embedding"]

            print(f"\n[사진 확인]: {original_name}")
            print("0: 젊은층 (10~40세) / 1: 50대 이상 (도움 대상)")
            answer = input("번호 입력 (0 또는 1): ")
            while answer not in ['0', '1']:
                answer = input("0 또는 1만 입력하세요: ")

            X_new_data.append(face_feature)
            y_new_data.append(int(answer))

            # 실시간으로 전체 몇 장이 채워지고 있는지 누적 장수 계산해서 보여줌
            print(f"✔️ 현재 총 누적 장수: {current_total_count + len(X_new_data)}장째 수집 중...")

        except Exception as e:
            print(f"⚠️ 에러 발생: {e}. 얼굴이 잘 나온 다른 사진을 써주세요.")
        finally:
            if os.path.exists(temp_english_name):
                os.remove(temp_english_name)
            if os.path.exists(original_name):
                os.remove(original_name)

# =====================================================================
# [학습 및 저장 단계] 모델 학습 및 드라이브 업데이트
# =====================================================================
if len(X_new_data) > 0:
    X_new_arr = np.array(X_new_data)
    y_new_arr = np.array(y_new_data)

    print("\n🔄 기존 AI 가중치에 새 사진 데이터를 더해 누적 학습 진행 중...")

    # 기존 가중치 위에 새 사진들을 얹어서 모델을 업데이트합니다.
    kiosk_ai_model.fit(X_new_arr, y_new_arr)

    # 최종 업데이트된 총 장수 계산
    final_total_count = current_total_count + len(X_new_data)

    # 1. 모델 pkl 파일 내 구글 드라이브에 덮어쓰기 저장
    with open(model_filename, 'wb') as file:
        pickle.dump(kiosk_ai_model, file)

    # 2. 장수 기록용 txt 파일 내 구글 드라이브에 저장 (다음번에 켰을 때 이 숫자를 기억함)
    with open(count_filename, 'w') as f:
        f.write(str(final_total_count))

    print("==================================================")
    print(f"🎉 누적 완료! 총 {final_total_count}장의 패턴이 모두 담긴 AI 모델이 드라이브에 새로 저장되었습니다!")
    print("==================================================")
else:
    print("\n⚠️ 새로 업로드한 사진이 없어 모델을 업데이트하지 않고 기존 상태를 유지합니다.")

🎯 [기존 모델 불러오기 성공]
📂 파일 위치: /content/drive/MyDrive/kiosk_ai_model.pkl
📊 현재 이 AI 모델에 누적 저장된 기존 사진 개수: 70장

▶ 기존 70장에 이어서 추가할 [새 사진]을 등록합니다.
  (추가할 사진을 모두 올렸다면 '취소'나 '창닫기'를 누르세요)


Saving KakaoTalk_20260618_184432653.jpg to KakaoTalk_20260618_184432653.jpg


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5
To: /root/.deepface/weights/vgg_face_weights.h5


26-06-18 09:50:31 - 🔗 vgg_face_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/vgg_face_weights.h5 to /root/.deepface/weights/vgg_face_weights.h5...


100%|██████████| 580M/580M [00:04<00:00, 135MB/s] 


...

Saving KakaoTalk_20260618_184432653_17.jpg to KakaoTalk_20260618_184432653_17.jpg
Saving KakaoTalk_20260618_184432653_18.jpg to KakaoTalk_20260618_184432653_18.jpg
Saving KakaoTalk_20260618_184432653_19.jpg to KakaoTalk_20260618_184432653_19.jpg
Saving KakaoTalk_20260618_184432653_20.jpg to KakaoTalk_20260618_184432653_20.jpg
Saving KakaoTalk_20260618_184432653_21.jpg to KakaoTalk_20260618_184432653_21.jpg
Saving KakaoTalk_20260618_184432653_22.jpg to KakaoTalk_20260618_184432653_22.jpg
Saving KakaoTalk_20260618_184432653_23.jpg to KakaoTalk_20260618_184432653_23.jpg
Saving KakaoTalk_20260618_184432653_24.jpg to KakaoTalk_20260618_184432653_24.jpg
Saving KakaoTalk_20260618_184432653_25.jpg to KakaoTalk_20260618_184432653_25.jpg
Saving KakaoTalk_20260618_184432653_26.jpg to KakaoTalk_20260618_184432653_26.jpg
Saving KakaoTalk_20260618_184432653_27.jpg to KakaoTalk_20260618_184432653_27.jpg
Saving KakaoTalk_20260618_184432653_28.jpg to KakaoTalk_20260618_184432653_28.jpg
Saving KakaoTalk

Saving KakaoTalk_20260618_184442210_07 (1).jpg to KakaoTalk_20260618_184442210_07 (1).jpg
Saving KakaoTalk_20260618_184442210_08 (1).jpg to KakaoTalk_20260618_184442210_08 (1).jpg
Saving KakaoTalk_20260618_184442210_09 (1).jpg to KakaoTalk_20260618_184442210_09 (1).jpg
Saving KakaoTalk_20260618_184442210_10.jpg to KakaoTalk_20260618_184442210_10.jpg

[사진 확인]: KakaoTalk_20260618_184442210_07 (1).jpg
0: 젊은층 (10~40세) / 1: 50대 이상 (도움 대상)
번호 입력 (0 또는 1): 0
✔️ 현재 총 누적 장수: 108장째 수집 중...

[사진 확인]: KakaoTalk_20260618_184442210_08 (1).jpg
0: 젊은층 (10~40세) / 1: 50대 이상 (도움 대상)
번호 입력 (0 또는 1): 0
✔️ 현재 총 누적 장수: 109장째 수집 중...

[사진 확인]: KakaoTalk_20260618_184442210_09 (1).jpg
0: 젊은층 (10~40세) / 1: 50대 이상 (도움 대상)
번호 입력 (0 또는 1): 0
✔️ 현재 총 누적 장수: 110장째 수집 중...

[사진 확인]: KakaoTalk_20260618_184442210_10.jpg
0: 젊은층 (10~40세) / 1: 50대 이상 (도움 대상)
번호 입력 (0 또는 1): 0
✔️ 현재 총 누적 장수: 111장째 수집 중...


Saving KakaoTalk_20260618_184442210_11 (1).jpg to KakaoTalk_20260618_184442210_11 (1).jpg
Saving KakaoTalk_20260618_184442210_12 (1).jpg to KakaoTalk_20260618_184442210_12 (1).jpg
Saving KakaoTalk_20260618_184442210_13 (1).jpg to KakaoTalk_20260618_184442210_13 (1).jpg
Saving KakaoTalk_20260618_184442210_14 (1).jpg to KakaoTalk_20260618_184442210_14 (1).jpg
Saving KakaoTalk_20260618_184442210_15 (1).jpg to KakaoTalk_20260618_184442210_15 (1).jpg
Saving KakaoTalk_20260618_184442210_16 (1).jpg to KakaoTalk_20260618_184442210_16 (1).jpg
Saving KakaoTalk_20260618_184442210_19 (1).jpg to KakaoTalk_20260618_184442210_19 (1).jpg
Saving KakaoTalk_20260618_184442210_20 (1).jpg to KakaoTalk_20260618_184442210_20 (1).jpg
Saving KakaoTalk_20260618_184442210_23 (1).jpg to KakaoTalk_20260618_184442210_23 (1).jpg
Saving KakaoTalk_20260618_184442210_25 (1).png to KakaoTalk_20260618_184442210_25 (1).png

[사진 확인]: KakaoTalk_20260618_184442210_11 (1).jpg
0: 젊은층 (10~40세) / 1: 50대 이상 (도움 대상)
번호 입력 (0 또는 1)

Saving KakaoTalk_20260618_184442210_17.jpg to KakaoTalk_20260618_184442210_17.jpg
Saving KakaoTalk_20260618_184442210_21.jpg to KakaoTalk_20260618_184442210_21.jpg
Saving KakaoTalk_20260618_184442210_22 (1).jpg to KakaoTalk_20260618_184442210_22 (1).jpg
Saving KakaoTalk_20260618_184442210_24.png to KakaoTalk_20260618_184442210_24.png
Saving KakaoTalk_20260618_184442210_26.jpg to KakaoTalk_20260618_184442210_26.jpg
Saving KakaoTalk_20260618_184442210_27 (1).jpg to KakaoTalk_20260618_184442210_27 (1).jpg
Saving KakaoTalk_20260618_184442210_27.jpg to KakaoTalk_20260618_184442210_27.jpg
Saving KakaoTalk_20260618_184442210_28.png to KakaoTalk_20260618_184442210_28.png
Saving KakaoTalk_20260618_184442210_29.jpg to KakaoTalk_20260618_184442210_29.jpg
Saving KakaoTalk_20260618_184446888.jpg to KakaoTalk_20260618_184446888.jpg
Saving KakaoTalk_20260618_184446888_01.jpg to KakaoTalk_20260618_184446888_01.jpg
Saving KakaoTalk_20260618_184446888_02.jpg to KakaoTalk_20260618_184446888_02.jpg
Saving


▶ 사진 등록 완료! 이번에 총 66장의 사진이 새로 추가되었습니다.

🔄 기존 AI 가중치에 새 사진 데이터를 더해 누적 학습 진행 중...
🎉 누적 완료! 총 136장의 패턴이 모두 담긴 AI 모델이 드라이브에 새로 저장되었습니다!
